# V03 — Plotly Graph Objects Foundations

**`graph_objects` is the low-level API.** It gives you full control over every pixel of the chart — traces, layouts, markers, annotations, shapes. You need this when `px` can't do what you want.

**Mental model:** A Plotly figure = `go.Figure(data=[traces], layout=go.Layout(...))`.
Every chart element is an object with its own properties.

**Reference:** [Graph Objects docs](https://plotly.com/python/graph-objects/)

**Allowed:** `plotly.graph_objects`, `plotly.express`, `pandas`, `numpy`


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from sklearn.datasets import fetch_openml, fetch_california_housing

# --- Datasets ---
retail_raw = fetch_openml(name='onlineretail', version=1, as_frame=True, parser='auto').frame
retail = retail_raw.copy()
retail.columns = [c.strip() for c in retail.columns]
retail['InvoiceDate'] = pd.to_datetime(retail['InvoiceDate'])
retail['Quantity'] = pd.to_numeric(retail['Quantity'], errors='coerce')
retail['UnitPrice'] = pd.to_numeric(retail['UnitPrice'], errors='coerce')
retail = retail[retail['Quantity'] > 0].dropna(subset=['CustomerID'])
retail['Revenue'] = retail['Quantity'] * retail['UnitPrice']
retail['Month'] = retail['InvoiceDate'].dt.to_period('M').astype(str)

monthly = retail.groupby('Month').agg(
    Revenue=('Revenue','sum'),
    Orders=('InvoiceNo','nunique'),
    Customers=('CustomerID','nunique')
).reset_index()

housing_raw = fetch_california_housing(as_frame=True)
housing = housing_raw.frame.copy()
housing.columns = [c.lower() for c in housing.columns]
housing_sample = housing.sample(2000, random_state=42)

# Credit dataset for categorical charts
credit_raw = fetch_openml(name='credit-g', version=1, as_frame=True, parser='auto').frame
credit = credit_raw.copy()
credit['credit_amount'] = pd.to_numeric(credit['credit_amount'], errors='coerce')
credit['duration'] = pd.to_numeric(credit['duration'], errors='coerce')
credit['age'] = pd.to_numeric(credit['age'], errors='coerce')

print(f"Monthly: {monthly.shape} | Housing sample: {housing_sample.shape} | Credit: {credit.shape}")

---
## Exercise 1 — go.Scatter: Full Marker Control

**Spec:** Build a scatter plot from scratch using `go.Figure` and `go.Scatter` — no `px`.
- X: `medinc`, Y: `medhousval` from `housing_sample`
- Marker: `size=5`, `color=housing_sample['houseage']`, `colorscale='Plasma'`, `showscale=True`, `colorbar=dict(title='House Age')`
- `mode='markers'`
- `opacity=0.6`
- Hover: `hovertemplate='Income: %{x:.2f}<br>Value: %{y:.2f}<br>Age: %{marker.color:.0f}<extra></extra>'`
- Layout: title `'Income vs House Value (colored by age)'`, xaxis title `'Median Income'`, yaxis title `'Median House Value'`
- Set `plot_bgcolor='#F8F9FA'`, `paper_bgcolor='white'`
- Add gridlines: `xaxis_gridcolor='#E0E0E0'`, `yaxis_gridcolor='#E0E0E0'`
- Assign to `fig1`

In [ ]:
# YOUR CODE HERE — use go.Figure and go.Scatter only
fig1 = None

fig1.show()

In [ ]:
# --- ASSERTIONS ---
assert fig1.layout.title.text == 'Income vs House Value (colored by age)'
assert fig1.data[0].type == 'scatter'
assert fig1.data[0].mode == 'markers'
assert fig1.data[0].marker.colorscale == 'Plasma' or 'plasma' in str(fig1.data[0].marker.colorscale).lower()
assert fig1.data[0].marker.showscale == True
assert fig1.layout.plot_bgcolor == '#F8F9FA'
assert fig1.layout.xaxis.gridcolor == '#E0E0E0'
assert 'Income' in fig1.data[0].hovertemplate
print("✓ Exercise 1 passed")

---
## Exercise 2 — go.Bar: Stacked with Totals

**Spec:** Stacked bar chart of SaaS-style revenue by purpose and credit class, with total annotations on top.

Build `purpose_class`: crosstab of `purpose` (top 6 by count) vs `class` in `credit`, values = mean `credit_amount`.

- Create one `go.Bar` trace per credit class (`'good'`, `'bad'`)
- `barmode='stack'` on the layout
- Colors: good = `'#4CAF50'`, bad = `'#F44336'`
- Add annotations above each bar showing the total (sum of both classes)
- Annotations: `font_size=9`, `showarrow=False`, `yanchor='bottom'`
- Title: `'Mean Credit Amount by Purpose and Credit Class'`
- Assign to `fig2`

In [ ]:
top6_purpose = credit['purpose'].value_counts().nlargest(6).index
purpose_class = (
    credit[credit['purpose'].isin(top6_purpose)]
    .groupby(['purpose', 'class'])['credit_amount']
    .mean().round(0).unstack('class')
    .fillna(0)
)

# YOUR CODE HERE
fig2 = None

fig2.show()

In [ ]:
# --- ASSERTIONS ---
assert fig2.layout.title.text == 'Mean Credit Amount by Purpose and Credit Class'
assert fig2.layout.barmode == 'stack'
assert len(fig2.data) == 2
trace_names = {t.name for t in fig2.data}
assert 'good' in trace_names and 'bad' in trace_names
assert fig2.data[0].marker.color in ('#4CAF50', '#F44336')
# Annotations for totals
assert len(fig2.layout.annotations) == 6, "One total annotation per purpose"
print("✓ Exercise 2 passed")

---
## Exercise 3 — go.Heatmap: Correlation Matrix

**Spec:** Build a publication-quality correlation heatmap from scratch.

Compute `corr_matrix` from housing numeric columns: `['medinc', 'houseage', 'averooms', 'avebedrms', 'population', 'aveoccup', 'medhousval']`

- Use `go.Heatmap`
- `z=corr_matrix.values`, `x=cols`, `y=cols`
- `colorscale='RdBu'`, `zmid=0`, `zmin=-1`, `zmax=1`
- `text`: correlation values rounded to 2dp
- `texttemplate='%{text}'`, `textfont_size=10`
- `hoverongaps=False`
- Mask upper triangle: set values above diagonal to `None`
- Title: `'Housing Feature Correlation Matrix'`
- Square aspect ratio: `yaxis_scaleanchor='x'`
- Assign to `fig3`

In [ ]:
cols = ['medinc', 'houseage', 'averooms', 'avebedrms', 'population', 'aveoccup', 'medhousval']
corr_matrix = housing[cols].corr().round(2)

# Mask upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
masked_corr = corr_matrix.copy().astype(float)
masked_corr[mask] = None

# YOUR CODE HERE
fig3 = None

fig3.show()

In [ ]:
# --- ASSERTIONS ---
assert fig3.layout.title.text == 'Housing Feature Correlation Matrix'
assert fig3.data[0].type == 'heatmap'
assert fig3.data[0].zmid == 0
assert fig3.data[0].zmin == -1 and fig3.data[0].zmax == 1
# Upper triangle should be None
z = np.array(fig3.data[0].z, dtype=object)
assert z[0][1] is None or (isinstance(z[0][1], float) and np.isnan(z[0][1])), \
    "Upper triangle must be masked"
assert fig3.layout.yaxis.scaleanchor == 'x'
print("✓ Exercise 3 passed")

**Interpretation:** *(Which features are most correlated with house value? Any surprising correlations?)*

---
## Exercise 4 — Annotations & Shapes: Annotated Time Series

**Spec:** Build an annotated revenue line chart that calls out key events.

- Use `go.Scatter` for the monthly revenue line
- `mode='lines+markers'`, line color `'#1565C0'`, line width 2
- Fill under line: `fill='tozeroy'`, `fillcolor='rgba(21, 101, 192, 0.1)'`
- Find the month with max revenue and add:
  - A `go.layout.Shape` (vertical line) at that x position: `line_color='red'`, `line_dash='dash'`
  - An annotation: `text='Peak Revenue'`, `arrowhead=2`, `arrowcolor='red'`
- Find the month with min revenue (exclude first month) and annotate similarly with `'Revenue Trough'` in orange
- Add a horizontal shape (rectangle) highlighting the last 3 months: `fillcolor='rgba(255,235,59,0.15)'`, `line_width=0`
- Title: `'Monthly Revenue with Key Events Annotated'`
- Assign to `fig4`

In [ ]:
peak_month = monthly.loc[monthly['Revenue'].idxmax(), 'Month']
trough_month = monthly.loc[monthly.iloc[1:]['Revenue'].idxmin(), 'Month']
last_3 = monthly['Month'].iloc[-3:].tolist()

# YOUR CODE HERE
fig4 = None

fig4.show()

In [ ]:
# --- ASSERTIONS ---
assert fig4.layout.title.text == 'Monthly Revenue with Key Events Annotated'
assert fig4.data[0].fill == 'tozeroy'
# At least 2 annotations (peak + trough)
assert len(fig4.layout.annotations) >= 2
annotation_texts = [a.text for a in fig4.layout.annotations]
assert 'Peak Revenue' in annotation_texts
assert 'Revenue Trough' in annotation_texts
# At least 3 shapes: 2 vlines + 1 rectangle
assert len(fig4.layout.shapes) >= 3
shape_types = [s.type for s in fig4.layout.shapes]
assert 'line' in shape_types
assert 'rect' in shape_types
print("✓ Exercise 4 passed")

---
## Exercise 5 — go.Indicator: KPI Tiles

**Spec:** Build 3 KPI indicator tiles in a single figure using `go.Indicator`.

Compute from `monthly`:
- Total Revenue (sum), with delta vs previous month
- Total Orders (sum), with delta vs previous month  
- Average Order Value = Revenue / Orders (last month vs previous month)

For each indicator:
- `mode='number+delta+gauge'`
- `delta_reference`: previous period value
- `gauge_axis_range=[0, max_value * 1.2]`
- Positive delta color: `'#4CAF50'`, Negative: `'#F44336'`
- Use `domain` to place 3 tiles side by side: x=[0,0.33], [0.34,0.66], [0.67,1.0]
- Title: `'Retail KPI Dashboard — Latest Month'`
- Assign to `fig5`

In [ ]:
last = monthly.iloc[-1]
prev = monthly.iloc[-2]

# YOUR CODE HERE
fig5 = None

fig5.show()

In [ ]:
# --- ASSERTIONS ---
assert fig5.layout.title.text == 'Retail KPI Dashboard — Latest Month'
assert len(fig5.data) == 3, "Must have 3 indicator traces"
for trace in fig5.data:
    assert trace.type == 'indicator'
    assert 'number' in trace.mode
    assert 'delta' in trace.mode
    assert trace.delta.reference is not None
# Tiles must be side by side (different x domains)
domains_x = [tuple(t.domain.x) for t in fig5.data]
assert len(set(domains_x)) == 3, "Each indicator must have a unique domain"
print("✓ Exercise 5 passed")

---
## Exercise 6 — go.Pie with Custom Pull & Text

**Spec:** Build a donut chart of revenue share by credit purpose.

- Use `go.Pie`
- `hole=0.45` (donut style)
- `labels`: top 8 purposes by mean credit amount
- `values`: mean credit_amount per purpose
- `pull`: pull the largest slice by 0.1 (compute which is largest)
- `textinfo='label+percent'`
- `textposition='outside'`
- `marker_colors`: use `px.colors.qualitative.Set3`
- `hovertemplate='<b>%{label}</b><br>Avg Credit: $%{value:,.0f}<br>Share: %{percent}<extra></extra>'`
- Add center annotation: text `'Credit\nPurpose'`, font size 14, in center of donut
- Title: `'Mean Credit Amount by Purpose'`
- Assign to `fig6`

In [ ]:
purpose_rev = (
    credit.groupby('purpose')['credit_amount']
    .mean().round(0).nlargest(8)
    .reset_index()
)
largest_idx = purpose_rev['credit_amount'].idxmax()
pull_vals = [0.1 if i == largest_idx else 0 for i in range(len(purpose_rev))]

# YOUR CODE HERE
fig6 = None

fig6.show()

In [ ]:
# --- ASSERTIONS ---
assert fig6.layout.title.text == 'Mean Credit Amount by Purpose'
assert fig6.data[0].type == 'pie'
assert fig6.data[0].hole == 0.45
assert list(fig6.data[0].pull).count(0.1) == 1, "Only one slice should be pulled"
assert fig6.data[0].textposition == 'outside'
center_annot = [a for a in fig6.layout.annotations if 'Credit' in a.text]
assert len(center_annot) >= 1, "Center annotation missing"
print("✓ Exercise 6 passed")

---
## Exercise 7 — go.Candlestick: Financial OHLC Chart

**Spec:** Build a synthetic OHLC candlestick chart — standard in financial analytics.

Generate synthetic daily price data:
- 90 trading days, starting 2023-01-01
- Use `np.random.seed(42)`, random walk: `close[t] = close[t-1] * (1 + np.random.normal(0.001, 0.02))`
- Open = previous close, High = max(open, close) * uniform(1.001, 1.02), Low = min(open, close) * uniform(0.98, 0.999)

- Use `go.Candlestick`
- Increasing candles: `line_color='#26A69A'`, `fillcolor='#26A69A'`
- Decreasing candles: `line_color='#EF5350'`, `fillcolor='#EF5350'`
- Add a 20-day SMA line as `go.Scatter` overlay: color `'#FF9800'`, width 1.5
- Add a 5-day SMA: color `'#2196F3'`, width 1
- Remove range slider: `xaxis_rangeslider_visible=False`
- Title: `'Synthetic Stock Price (90 Days)'`
- Assign to `fig7`

In [ ]:
np.random.seed(42)
n_days = 90
dates = pd.date_range('2023-01-01', periods=n_days, freq='B')
close = [100.0]
for _ in range(n_days - 1):
    close.append(close[-1] * (1 + np.random.normal(0.001, 0.02)))
close = np.array(close)
open_ = np.concatenate([[close[0]], close[:-1]])
high = np.maximum(open_, close) * np.random.uniform(1.001, 1.02, n_days)
low = np.minimum(open_, close) * np.random.uniform(0.98, 0.999, n_days)
ohlc = pd.DataFrame({'Date': dates, 'Open': open_, 'High': high, 'Low': low, 'Close': close})
ohlc['SMA5'] = ohlc['Close'].rolling(5).mean()
ohlc['SMA20'] = ohlc['Close'].rolling(20).mean()

# YOUR CODE HERE
fig7 = None

fig7.show()

In [ ]:
# --- ASSERTIONS ---
assert fig7.layout.title.text == 'Synthetic Stock Price (90 Days)'
trace_types = [t.type for t in fig7.data]
assert 'candlestick' in trace_types
assert trace_types.count('scatter') >= 2, "Must have at least 2 SMA lines"
assert fig7.layout.xaxis.rangeslider.visible == False
candle = [t for t in fig7.data if t.type == 'candlestick'][0]
assert candle.increasing.line.color == '#26A69A'
assert candle.decreasing.line.color == '#EF5350'
print("✓ Exercise 7 passed")

**Interpretation:** *(What is the overall price trend? When do the SMAs cross — and what does that signal?)*

---
## Exercise 8 — go.Sankey: Flow Diagram

**Spec:** Build a Sankey diagram showing customer journey stages.

Use this flow data:
```
Visitor → Registered: 10000
Visitor → Bounced: 40000
Registered → Active: 6000
Registered → Churned: 4000
Active → Converted: 3500
Active → Lapsed: 2500
Converted → Retained: 2800
Converted → Lost: 700
```
- Use `go.Sankey`
- Node colors: use `px.colors.qualitative.Plotly` (one per unique node)
- Link colors: `rgba` version of source node color at 40% opacity
- `node_pad=20`, `node_thickness=25`
- `arrangement='snap'`
- Title: `'Customer Journey Flow'`
- Assign to `fig8`

In [ ]:
nodes = ['Visitor', 'Registered', 'Bounced', 'Active', 'Churned',
         'Converted', 'Lapsed', 'Retained', 'Lost']
node_idx = {n: i for i, n in enumerate(nodes)}

flows = [
    ('Visitor', 'Registered', 10000), ('Visitor', 'Bounced', 40000),
    ('Registered', 'Active', 6000), ('Registered', 'Churned', 4000),
    ('Active', 'Converted', 3500), ('Active', 'Lapsed', 2500),
    ('Converted', 'Retained', 2800), ('Converted', 'Lost', 700)
]

# YOUR CODE HERE
fig8 = None

fig8.show()

In [ ]:
# --- ASSERTIONS ---
assert fig8.layout.title.text == 'Customer Journey Flow'
assert fig8.data[0].type == 'sankey'
assert len(fig8.data[0].node.label) == len(nodes)
assert len(fig8.data[0].link.value) == len(flows)
assert fig8.data[0].node.pad == 20
assert fig8.data[0].node.thickness == 25
assert fig8.data[0].arrangement == 'snap'
print("✓ Exercise 8 passed")

**Interpretation:** *(What is the overall conversion rate from Visitor to Retained? Where is the biggest drop-off?)*

---
## Exercise 9 — go.Scattergeo: Bubble Map

**Spec:** Build a geographic bubble map of retail revenue.

Use the country_revenue + coordinates data below.
- Use `go.Scattergeo`
- `mode='markers'`
- Marker size: scale revenue to [5, 50] using min-max scaling
- Marker color: Revenue, colorscale `'YlOrRd'`, `showscale=True`
- `text`: `Country: $Revenue` on hover
- Projection: `'natural earth'`
- Title: `'Retail Revenue by Country (Bubble Map)'`
- Assign to `fig9`

In [ ]:
country_coords = {
    'United Kingdom': (55.4, -3.4), 'Germany': (51.2, 10.4),
    'France': (46.2, 2.2), 'EIRE': (53.1, -8.2),
    'Spain': (40.5, -3.7), 'Netherlands': (52.1, 5.3),
    'Belgium': (50.5, 4.5), 'Switzerland': (46.8, 8.2),
    'Portugal': (39.4, -8.2), 'Australia': (-25.3, 133.8),
    'Norway': (60.5, 8.5), 'Sweden': (60.1, 18.6),
    'Denmark': (56.3, 9.5), 'Japan': (36.2, 138.3), 'Finland': (61.9, 25.7)
}

country_revenue = (
    retail.groupby('Country')['Revenue'].sum().reset_index()
    .query('Country in @country_coords')
)
country_revenue['lat'] = country_revenue['Country'].map(lambda c: country_coords[c][0])
country_revenue['lon'] = country_revenue['Country'].map(lambda c: country_coords[c][1])
rev_min, rev_max = country_revenue['Revenue'].min(), country_revenue['Revenue'].max()
country_revenue['size_scaled'] = 5 + 45 * (country_revenue['Revenue'] - rev_min) / (rev_max - rev_min)

# YOUR CODE HERE
fig9 = None

fig9.show()

In [ ]:
# --- ASSERTIONS ---
assert fig9.layout.title.text == 'Retail Revenue by Country (Bubble Map)'
assert fig9.data[0].type == 'scattergeo'
assert fig9.layout.geo.projection.type == 'natural earth'
sizes = list(fig9.data[0].marker.size)
assert min(sizes) >= 4 and max(sizes) <= 51, "Sizes must be in [5,50] range"
assert fig9.data[0].marker.showscale == True
print("✓ Exercise 9 passed")

---
## Exercise 10 — Capstone: Custom Theme Engine

**Spec:** Build a reusable chart theme function that applies a consistent corporate style to any Plotly figure.

Write `apply_corporate_theme(fig, title=None, subtitle=None)` that:
1. Sets `paper_bgcolor='#FAFAFA'`, `plot_bgcolor='white'`
2. Sets font: `family='Arial'`, `size=12`, `color='#333333'`
3. Sets gridlines: light grey (`#EEEEEE`), width 1, both axes
4. Sets title font size 16, bold, color `'#1A237E'`
5. If `subtitle` provided: adds it as an annotation below the title, font size 11, color `'#666666'`
6. Adds a footer annotation: `'Source: Internal Analytics'` at bottom-left
7. Sets legend: `bgcolor='rgba(255,255,255,0.8)'`, `bordercolor='#CCCCCC'`, `borderwidth=1`
8. Returns the modified figure

Then apply to `fig4` (time series) and verify all properties.

In [ ]:
def apply_corporate_theme(fig: go.Figure,
                           title: str = None,
                           subtitle: str = None) -> go.Figure:
    """
    Apply consistent corporate styling to any Plotly figure.
    Returns modified figure.
    """
    # YOUR CODE HERE
    pass

import copy
fig10 = apply_corporate_theme(
    copy.deepcopy(fig4),
    title='Monthly Revenue Trend',
    subtitle='FY2023 | UK Retail Operations'
)
fig10.show()

In [ ]:
# --- ASSERTIONS ---
assert fig10.layout.paper_bgcolor == '#FAFAFA'
assert fig10.layout.plot_bgcolor == 'white'
assert fig10.layout.font.family == 'Arial'
assert fig10.layout.font.color == '#333333'
assert fig10.layout.xaxis.gridcolor == '#EEEEEE'
assert fig10.layout.yaxis.gridcolor == '#EEEEEE'
assert fig10.layout.title.font.size == 16
assert fig10.layout.title.font.color == '#1A237E'
# Footer annotation
annotation_texts = [a.text for a in fig10.layout.annotations]
assert any('Source' in t for t in annotation_texts), "Footer annotation missing"
# Subtitle
assert any('FY2023' in t for t in annotation_texts), "Subtitle annotation missing"
assert fig10.layout.legend.bordercolor == '#CCCCCC'
print("✓ Exercise 10 passed")